In [3]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
os.makedirs("model", exist_ok=True)
print("Directory 'model/' created/verified successfully.")

Directory 'model/' created/verified successfully.


In [5]:
# Set random seed for reproducibility
np.random.seed(42)
n_samples = 2500

# Generating 12 Input Features
X_raw = pd.DataFrame({
    'credit_score': np.random.randint(350, 850, size=n_samples),
    'age': np.random.randint(18, 70, size=n_samples),
    'tenure': np.random.randint(0, 10, size=n_samples),
    'balance': np.random.uniform(0, 250000, size=n_samples),
    'num_of_products': np.random.randint(1, 4, size=n_samples),
    'has_cr_card': np.random.choice([0, 1], size=n_samples),
    'is_active_member': np.random.choice([0, 1], size=n_samples),
    'estimated_salary': np.random.uniform(10000, 150000, size=n_samples),
    'country_code': np.random.choice([1, 2, 3], size=n_samples),
    'satisfaction_score': np.random.randint(1, 6, size=n_samples),
    'complain_registered': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]),
    'points_earned': np.random.randint(100, 1000, size=n_samples)
})

# Define target variable mapping with non-linear relationships
y_prob = 1 / (1 + np.exp(-(
    -0.002 * X_raw['credit_score'] + 
    0.04 * X_raw['age'] - 
    0.8 * X_raw['is_active_member'] + 
    1.5 * X_raw['complain_registered'] - 0.5
)))
y = (y_prob > 0.45).astype(int)

# Split into Train (70%) and Test (30%) splits
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.3, random_state=42, stratify=y
)

# Save test dataset including target column as requested by instructions
test_df = X_test.copy()
test_df['target'] = y_test
test_df.to_csv("test_data.csv", index=False)

print(f"Dataset Shape - Features: {X_raw.shape[1]}, Instances: {X_raw.shape[0]}")
print(f"Test data saved successfully as 'test_data.csv' with {len(X_test)} rows.")

Dataset Shape - Features: 12, Instances: 2500
Test data saved successfully as 'test_data.csv' with 750 rows.


In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler artifact
joblib.dump(scaler, "model/scaler.pkl")
print("StandardScaler fitted and saved to 'model/scaler.pkl'.")

StandardScaler fitted and saved to 'model/scaler.pkl'.


In [7]:
# Define the 5 mandatory classification models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

model_filenames = {
    'Logistic Regression': 'model/log_reg_model.pkl',
    'Decision Tree': 'model/dt_model.pkl',
    'KNN': 'model/knn_model.pkl',
    'Naive Bayes': 'model/nb_model.pkl',
    'Support Vector Machine': 'model/svm_model.pkl'
}

evaluation_results = []

for name, model in models.items():
    # Tree models perform well without scaling; distance/gradient models use scaled data
    if name in ['Logistic Regression', 'KNN', 'Support Vector Machine', 'Naive Bayes']:
        model.fit(X_train_scaled, y_train)
        X_eval = X_test_scaled
    else:
        model.fit(X_train, y_train)
        X_eval = X_test.values
        
    # Save trained model to disk
    joblib.dump(model, model_filenames[name])
    
    # Predict outcomes and probabilities
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1] if hasattr(model, "predict_proba") else y_pred
    
    # Compute all 6 required evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    evaluation_results.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

# Convert results into a DataFrame and display comparison table
comparison_df = pd.DataFrame(evaluation_results)
print("--- MODEL EVALUATION COMPARISON TABLE ---")
display(comparison_df)

C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


--- MODEL EVALUATION COMPARISON TABLE ---


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9947,0.9999,0.9974,0.9923,0.9949,0.9893
1,Decision Tree,0.9627,0.9795,0.9642,0.9642,0.9642,0.9252
2,KNN,0.9080,0.9688,0.9215,0.9003,0.9107,0.8161
3,Naive Bayes,0.8373,0.9860,0.9856,0.6982,0.8174,0.7112
4,Support Vector Machine,0.9720,0.9960,0.9793,0.9668,0.9730,0.9440


In [8]:
expected_files = [
    "test_data.csv",
    "model/scaler.pkl",
    "model/log_reg_model.pkl",
    "model/dt_model.pkl",
    "model/knn_model.pkl",
    "model/nb_model.pkl",
    "model/svm_model.pkl"
]

print("Verification Report:")
for file in expected_files:
    exists = os.path.exists(file)
    status = "✅ Found" if exists else "❌ Missing"
    print(f" - {file}: {status}")

Verification Report:
 - test_data.csv: ✅ Found
 - model/scaler.pkl: ✅ Found
 - model/log_reg_model.pkl: ✅ Found
 - model/dt_model.pkl: ✅ Found
 - model/knn_model.pkl: ✅ Found
 - model/nb_model.pkl: ✅ Found
 - model/svm_model.pkl: ✅ Found
